# Experiment 3: Neural Networks

2026 Autumn

## Description

Implement CNNs and a small Vision Transformer on CIFAR-10. Compare their results with data augmentation, BatchNorm, dropout and positional embeddings.

Complete the code between `########## Start/End of Your Code ##########`. Add comments where needed. Training and plotting helpers are provided.


### Environment and training settings

Use Python 3.10-3.12, PyTorch, torchvision, NumPy, matplotlib and Jupyter. A tested combination is Python 3.11, torch 2.5.1 and torchvision 0.20.1. For a new environment, run `python -m pip install torch==2.5.1 torchvision==0.20.1 numpy matplotlib jupyter`.

All students use the **low configuration**: **5,000 training images, 1,000 validation images and 5 epochs per run**. Use the official 10,000-image test set. Batch size is 64 and the random seed is 42. Complete five runs across Tasks 1-4.

A GPU is recommended when available. The code automatically uses CUDA when available and otherwise runs on CPU. Keep `num_workers=0` for Windows and laptop use. Run the tasks in order. The first run downloads about 170 MB of data, which is reused later. Allow about 1 GB of free disk space for data and local results. Download time is separate from training time.

CNNs use SGD with learning rate 0.01 and momentum 0.9. ViTs use AdamW with learning rate 0.001 and weight decay 0.01. All models use cross-entropy. The settings are fixed and no accuracy threshold is required.


In [ ]:
# Import necessary libraries. Additional libraries may be added if necessary.
import os, random, time, copy, json, csv, platform
from pathlib import Path
import numpy as np
import torch
import torchvision
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt

MODE = "low"  # Fixed course configuration.
SEED = 42
BATCH_SIZE = 64
DATA_ROOT = Path(os.environ.get("EXP3_DATA_ROOT", "./data"))
OUTPUT_ROOT = Path("./results_2026")
OUTPUT_ROOT.mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

N_TRAIN, N_VAL, EPOCHS = 5000, 1000, 5
seed_everything()
print({"mode": MODE, "seed": SEED, "train": N_TRAIN, "val": N_VAL,
       "epochs": EPOCHS, "batch_size": BATCH_SIZE, "device": str(device),
       "python": platform.python_version(), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "numpy": np.__version__})
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## Task 1: Implement Your First Neural Network

Implement the CNN below and complete the training and evaluation functions.

### Data Loading

CIFAR-10 contains 50,000 training images and 10,000 test images in 10 classes. Each image has size 32 x 32 with three colour channels.

Use the supplied split of 5,000 training and 1,000 validation images. It samples 500 training and 100 validation images per class with seed 42. The sets do not overlap. All tasks use this same split and normalize each channel with mean 0.5 and standard deviation 0.5. Validation and test images use no random augmentation.


In [ ]:
default_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)
])
raw_train = datasets.CIFAR10(root=str(DATA_ROOT), train=True, download=True)
targets = np.asarray(raw_train.targets)
rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for label in range(10):
    indices = rng.permutation(np.flatnonzero(targets == label))
    # Keep the established split, taking only the fixed low subset.
    val_indices.extend(indices[:100].tolist())
    train_indices.extend(indices[500:1000].tolist())
assert len(set(train_indices) & set(val_indices)) == 0
assert len(train_indices) == N_TRAIN and len(val_indices) == N_VAL
classes = raw_train.classes

# Retain only the selected images to reduce RAM usage on laptops.
def selected_dataset(indices):
    subset = copy.copy(raw_train)
    subset.data = raw_train.data[indices].copy()
    subset.targets = [raw_train.targets[i] for i in indices]
    subset.transform = default_transform
    return subset

train_base = selected_dataset(train_indices)
val_base = selected_dataset(val_indices)
del raw_train

def make_loaders(train_transform):
    train_data = copy.copy(train_base)
    train_data.transform = train_transform
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                              generator=generator, num_workers=0)
    val_loader = DataLoader(val_base, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, val_loader

print(f"Training: {len(train_base)}, validation: {len(val_base)}")
print("Image shape:", train_base[0][0].shape)


### Implementation

Use three convolutional blocks and two fully connected layers. This is the common CNN structure for Tasks 1-3. Task 1 does not use BatchNorm or dropout. Keep biases in convolutional and linear layers and return raw logits.

| Layer | Hyper-parameters | Output shape without batch |
|---|---|---|
| conv_1 block | Conv 3 to 32, k=5, s=1, p=2, ReLU, AvgPool(2,2) | 32 x 16 x 16 |
| conv_2 block | Conv 32 to 64, k=3, s=1, p=1, ReLU, AvgPool(2,2) | 64 x 8 x 8 |
| conv_3 block | Conv 64 to 128, k=3, s=1, p=1, ReLU, AvgPool(2,2) | 128 x 4 x 4 |
| flatten | Keep the batch dimension | 2048 |
| fc_1 | 2048 to 256, then ReLU | 256 |
| fc_2 | 256 to 10 | 10 |


In [ ]:
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete the specified layers or forward pass")
        ########## End of Your Code ##########

    def forward(self, x):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete the specified layers or forward pass")
        ########## End of Your Code ##########


### Training

Complete `train_one_epoch` and `evaluate`. Use `model.train()` during training. Use `model.eval()` and `torch.no_grad()` during evaluation. Compute mean loss over all examples, including the last batch.

Return `(mean_loss, accuracy)` from training and `(mean_loss, accuracy, ground_truths, predictions)` from evaluation. Report accuracy as a fraction from 0 to 1.

Run the supplied helper for five epochs and plot the training and validation curves. It keeps the checkpoint with the highest validation accuracy for final testing.


In [ ]:
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    ########## Start of Your Code ##########
    raise NotImplementedError("Return sample-weighted mean loss and accuracy")
    ########## End of Your Code ##########

def evaluate(model, loader):
    ########## Start of Your Code ##########
    raise NotImplementedError("Return loss, accuracy, true labels, predicted labels")
    ########## End of Your Code ##########


In [ ]:
# Provided helpers: reuse these in every task.
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize()

def fit(name, factory, train_transform, optimizer_kind="sgd"):
    seed_everything()
    model = factory().to(device)
    # Reset training RNG after construction so paired runs have the same RNG start.
    seed_everything()
    train_loader, val_loader = make_loaders(train_transform)
    optimizer = (optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
                 if optimizer_kind == "sgd" else
                 optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=0.001, weight_decay=0.01))
    history = []
    best_acc, best_epoch, best_state = -1.0, 0, None
    sync_device()
    start = time.perf_counter()
    for epoch in range(1, EPOCHS+1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader)
        history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                            val_loss=val_loss, val_acc=val_acc))
        if val_acc > best_acc:
            best_acc, best_epoch = val_acc, epoch
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(f"{name} epoch {epoch}/{EPOCHS}: loss {train_loss:.4f}, train {train_acc:.3f}, val {val_acc:.3f}")
    sync_device()
    elapsed = time.perf_counter() - start
    model.load_state_dict(best_state)
    model.eval()
    torch.save(best_state, OUTPUT_ROOT / (name + ".pt"))
    (OUTPUT_ROOT / (name + "_history.json")).write_text(json.dumps(history, indent=2), encoding="utf-8")
    result = dict(name=name, model=model.cpu(), history=history, best_epoch=best_epoch,
                  best_val_acc=best_acc, parameters=count_parameters(model), seconds=elapsed)
    return result

def plot_history(result):
    h = result["history"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for key, label in [("train", "Training"), ("val", "Validation")]:
        axes[0].plot([r["epoch"] for r in h], [r[key+"_loss"] for r in h], label=label)
        axes[1].plot([r["epoch"] for r in h], [r[key+"_acc"] for r in h], label=label)
    for ax, ylabel in zip(axes, ["Cross-entropy", "Accuracy"]):
        ax.set(xlabel="Epoch", ylabel=ylabel)
        ax.legend()
        ax.grid(alpha=0.2)
    fig.suptitle(result["name"] + f" | best validation epoch: {result['best_epoch']}")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / (result["name"] + "_curves.png"), dpi=150)
    plt.show()

results = {}
# Lightweight shape check before downloading/training further models.
assert MyNet()(torch.zeros(2, 3, 32, 32)).shape == (2, 10)
assert count_parameters(MyNet()) == 621898


In [ ]:
results["cnn"] = fit("cnn", MyNet, default_transform)
plot_history(results["cnn"])


### Testing

Use the shared testing cell after Task 4 to evaluate this model on the test set.


## Task 2: Implement Neural Network with Data Augmentation

Add `RandomCrop(32, padding=4)` and `RandomHorizontalFlip(p=0.5)` before `ToTensor` and normalization. Apply these transforms only to training images.

Keep the three-block CNN from Task 1 without BatchNorm or dropout. Train a fresh model for five epochs with the same initial weights and optimizer settings. Plot the curves and compare its test accuracy with Task 1.

### Data Loading and Implementation


In [ ]:
########## Start of Your Code ##########
transform_train = None  # Complete the required augmentation pipeline.
transform_test = default_transform
########## End of Your Code ##########
assert transform_train is not None


### Training


In [ ]:
results["cnn_aug"] = fit("cnn_aug", MyNet, transform_train)
plot_history(results["cnn_aug"])


### Testing

Use the shared final testing cell after Task 4.


## Task 3: Add BatchNorm and Dropout

Keep the convolutional layers, channel widths, pooling and fully connected layers from Task 2. Add BatchNorm between each convolution and ReLU. Add dropout before flattening and after the hidden fully connected ReLU, as shown below. Use the same augmentation, initial shared weights and training settings as Task 2.

### Implementation

| Layer | Hyper-parameters | Output shape without batch |
|---|---|---|
| conv_1 block | Conv 3 to 32, k=5, s=1, p=2, BatchNorm, ReLU, AvgPool(2,2) | 32 x 16 x 16 |
| conv_2 block | Conv 32 to 64, k=3, s=1, p=1, BatchNorm, ReLU, AvgPool(2,2) | 64 x 8 x 8 |
| conv_3 block | Conv 64 to 128, k=3, s=1, p=1, BatchNorm, ReLU, AvgPool(2,2) | 128 x 4 x 4 |
| dropout | nn.Dropout(p=0.25), then flatten | 2048 |
| fc_1 | 2048 to 256, ReLU, nn.Dropout(p=0.25) | 256 |
| fc_2 | 256 to 10, raw logits | 10 |

Keep biases in convolutional and linear layers. Use default affine BatchNorm2d. Train for five epochs and plot the curves.

`MyNet_drop` inherits the base layers from `MyNet`. Compare Task 2 and Task 3 to observe the combined effect of BatchNorm and dropout. No additional training runs are required.


In [ ]:
class MyNet_drop(MyNet):
    def __init__(self):
        # Reuse the exact convolutional and linear layers from Tasks 1-2.
        super().__init__()
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete the specified layers or forward pass")
        ########## End of Your Code ##########

    def forward(self, x):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete the specified layers or forward pass")
        ########## End of Your Code ##########


### Training


In [ ]:
assert MyNet_drop()(torch.zeros(2, 3, 32, 32)).shape == (2, 10)
results["cnn_bn_drop"] = fit("cnn_bn_drop", MyNet_drop, transform_train)
plot_history(results["cnn_bn_drop"])


### Testing

Use the shared final testing cell after Task 4.


## Task 4: Implement a Small Vision Transformer

Complete a small Vision Transformer using basic PyTorch layers and tensor operations. Do not use pre-trained models, ready-made attention or Transformer modules.

### Implementation

Use 8 x 8 patches to reduce CPU computation. The image becomes 16 patch tokens. Add one class token to obtain a sequence of length 17. Complete the forward passes of the attention layer, encoder block and ViT.

| Component | Specification |
|---|---|
| Patch projection | Conv2d 3 to 64, kernel=stride=8, padding=0 |
| Tokens | Flatten patches in row-major order, then prepend a learnable class token |
| Position | Learnable embeddings of shape (1,17,64) |
| Attention | 4 heads with dimension 16 per head, biased QKV and output projections |
| Attention weights | softmax(QK^T / sqrt(16)) along the key dimension, followed by dropout 0.1 |
| Encoder block | u = x + Dropout(MHSA(LayerNorm(x))), then y = u + MLP(LayerNorm(u)) |
| MLP | Linear(64,128), GELU, Dropout(0.1), Linear(128,64), Dropout(0.1) |
| Output | 2 encoder blocks, final LayerNorm, class-token Linear(64,10) |

Use default LayerNorm settings and raw output logits. Initialize class and position parameters with normal standard deviation 0.02. There is no embedding dropout. Apply dropout to the attention output once in its residual branch. The layer definitions are provided in the notebook.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim=64, heads=4, dropout=0.1):
        super().__init__()
        assert dim % heads == 0
        self.heads, self.head_dim = heads, dim // heads
        self.qkv = nn.Linear(dim, 3*dim)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete the specified forward pass")
        ########## End of Your Code ##########

class EncoderBlock(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim)
        self.residual_drop = nn.Dropout(0.1)
        self.mlp = nn.Sequential(nn.Linear(dim, 128), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(128, dim), nn.Dropout(0.1))

    def forward(self, x):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete the specified forward pass")
        ########## End of Your Code ##########

class TinyViT(nn.Module):
    def __init__(self, use_pos=True):
        super().__init__()
        self.use_pos = use_pos
        self.patch_embed = nn.Conv2d(3, 64, kernel_size=8, stride=8)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, 64))
        self.pos_embed = nn.Parameter(torch.zeros(1, 17, 64), requires_grad=use_pos)
        self.blocks = nn.Sequential(EncoderBlock(), EncoderBlock())
        self.norm = nn.LayerNorm(64)
        self.head = nn.Linear(64, 10)
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete the specified forward pass")
        ########## End of Your Code ##########


### Sanity checks

Run the supplied checks to confirm output shapes and finite gradients before training.


In [ ]:
seed_everything()
for factory in [MyNet, MyNet_drop, TinyViT]:
    model = factory()
    z = model(torch.randn(2, 3, 32, 32))
    assert z.shape == (2, 10) and torch.isfinite(z).all()
    F.cross_entropy(z, torch.tensor([0, 1])).backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all()
               for p in model.parameters() if p.requires_grad)
    print(factory.__name__, "parameters:", count_parameters(model), "shape/gradient: PASS")



### Training

Train one ViT with positional embeddings and one without them. Both use five epochs, the Task 2 augmentation and the same initialization and AdamW settings. The helper freezes and skips the position parameters in the second model.

Plot both learning curves. Include the two models in the final comparison table.


In [ ]:
results["vit_pos"] = fit("vit_pos", lambda: TinyViT(use_pos=True), transform_train, "adamw")
plot_history(results["vit_pos"])
results["vit_no_pos"] = fit("vit_no_pos", lambda: TinyViT(use_pos=False), transform_train, "adamw")
plot_history(results["vit_no_pos"])


### Testing

Run the following cell after all five models have finished training. It evaluates their validation-selected checkpoints on the test set and prints a comparison table.

Briefly summarize the observed differences in your report. Further tuning or extra training runs are not required.


In [ ]:
# Evaluate all five selected checkpoints after training is complete.
test_set = datasets.CIFAR10(str(DATA_ROOT), train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
summary = []
for name, result in results.items():
    model = result["model"].to(device)
    test_loss, accuracy, _, _ = evaluate(model, test_loader)
    summary.append(dict(name=name, train=N_TRAIN, val=N_VAL, test=len(test_set),
                        epochs=EPOCHS, best_epoch=result["best_epoch"],
                        parameters=result["parameters"], val_accuracy=result["best_val_acc"],
                        test_accuracy=accuracy, seconds=result["seconds"]))
    model.cpu()

print("name             params  val_acc test_acc train+val_seconds")
for row in summary:
    print(f"{row['name']:16s} {row['parameters']:7d} {row['val_accuracy']:.4f}  "
          f"{row['test_accuracy']:.4f}   {row['seconds']:.1f}")
with open(OUTPUT_ROOT / "summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")


## Requirements

- Complete Tasks 1-4 in PyTorch using the fixed low configuration.
- Keep code readable and retain the notebook outputs.
- Include the training curves, comparison table and a brief discussion in your report.
- Submit the completed `.ipynb` and `report.pdf` using the supplied report template. Datasets and model files are not required.
